# Milestone 3 Lab, notebook 5 of 6: A harness works the queue

## Where we are

| Notebook | What you do |
|---|---|
| 1. Where we are, and the problems by hand | Recall the question, find the data problems yourself, and correct the answer key |
| 2. Checks in code | Write two quality checks as queries, and prove each one works |
| 3. Quarantine, and the figures before and after | Set the wrong records aside, recalculate, and confirm the totals add up |
| 4. A to-do list and a helper, by hand | Ask a model to explain the problems, then build an agent that organises its own work |
| **5. A harness works the queue** (you are here) | Give every problem to a ready-made agent harness, and score its reports |
| 6. The same question, with real warnings | Ask the analyst the leader's question, now that the warnings come from real checks |

Model requests in this notebook: **about 40**. A full run of this lab costs about 10 US cents.

## What this notebook is for

In notebook 4 you built a to-do list, file tools, and a helper by hand. A **harness** is software that gives an agent all of those ready-made, along with things you did not build: searching files, editing them, and summarising a long conversation.

Here you use **Deep Agents**, LangChain's harness, on the whole queue of seven findings. You write **no tools**. You write two sets of instructions, say "investigate the findings", and watch.

Then the important part: you score its work against the causes we planted, and see what happens to a report that cannot be used.

In [3]:
import inspect
import json
import os
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display
from lab_helpers import OPENROUTER_BASE_URL, connect, load_result, show_code, show_doc
from langchain_openai import ChatOpenAI

from retail_profit_stack.quality.desk import build_desk, read_findings, read_reports, score_reports
from retail_profit_stack.quality.investigate import (
    HELPER_INSTRUCTIONS,
    MAIN_INSTRUCTIONS,
    build_investigator,
    run_investigator,
)
from retail_profit_stack.transform.pipeline import run_pipeline

client, MODEL = connect()

Connected to OpenRouter. Default model: anthropic/claude-haiku-5.5


## Step 1: The design

In [4]:
show_doc("HLD.md", "7. The investigator: an agent that organises its own work")

> **From `docs/HLD.md`: 7. The investigator: an agent that organises its own work**

In milestones 1 and 2, our code organised the agent: it gave it one question, or one task. Here the agent gets a folder and one instruction, "investigate every finding", and organises the work itself. Three ideas make that possible.

| Idea | What it means |
|---|---|
| **Files as workspace** | The agent reads and writes files in one folder, the desk. It cannot reach anything outside it |
| **To-do list** | The agent writes a list of what it must do and ticks items off. It is a tool that stores a list, so we can watch the plan |
| **Subagent (a helper)** | For each finding, the agent starts a fresh helper that handles that one finding and reports back in one line. Each helper starts clean, so one finding's detail does not crowd out the next |

Software that gives an agent these things ready-made is called a **harness**. The lab builds a small to-do list and helper by hand first, on three findings. Then it uses a ready-made harness, **Deep Agents** from LangChain, on the whole list.

## Step 2: The desk, with every finding on it

The same desk as in notebook 4, in a new folder, with all seven findings.

In [5]:
run_pipeline("sql")

DESK = Path("desk")
on_desk = build_desk(DESK)

print("Findings on the desk:", on_desk)

Findings on the desk: ['QF-001', 'QF-002', 'QF-003', 'QF-004', 'QF-005', 'QF-006', 'QF-007']


## Step 3: The instructions

There are two sets. The first is for the main agent, which runs the queue.

In [6]:
print(MAIN_INSTRUCTIONS)

You are a data engineer at Kestrel Ridge Outfitters. The quality checks found problems in the business's records, and you are working through them on your own.

Everything you need is in this folder:
- findings/ has one file for each finding
- rulebook.md is the data quality rulebook
- reports/ is where the reports go

Investigate every finding in findings/. Keep a to-do list with one item for each finding, and give each finding to the investigator subagent. Tell it only the finding id. It already knows how to investigate a finding, and it writes the report to reports/<finding id>.json.
When every finding has a .json report, write handover.md at the top of this folder: a short note for the data engineer. Group the findings that have the same cause, and say what to do first.
You cannot change the data or the outcome of a check. You only write notes.



The second is for the helper, which investigates one finding. You read these in notebook 4.

In [7]:
print(HELPER_INSTRUCTIONS)

You are a data engineer at Kestrel Ridge Outfitters. Investigate the one finding you are given.

Read the finding's file in findings/. Read rulebook.md for the rules of its check.
Look at the records in the file and choose the one cause whose sign matches them.

Write your report to reports/<finding id>.json with exactly these fields:
- finding_id
- likely_cause: one cause name, exactly as the rulebook writes it
- evidence: one or two sentences saying what in the records points to that cause
- suggested_action: one sentence for the data engineer

Then reply with one line: the finding id and the cause.



## Step 4: Build the agent

The model, as a LangChain object. `extra_body` asks OpenRouter to report what each call cost.

In [8]:
llm = ChatOpenAI(
    model=MODEL,
    base_url=OPENROUTER_BASE_URL,
    api_key=os.getenv("OPENROUTER_API_KEY"),
    timeout=120,
    max_retries=2,
    max_tokens=6000,
    extra_body={"usage": {"include": True}},
)

`build_investigator` is the production function that sets up the harness. Here is all of it. Compare it with what you wrote by hand:

- `subagents=[helper]` is your `investigate` tool.
- `TodoListMiddleware()` is your `write_todos` tool.
- `FilesystemBackend(root_dir=..., virtual_mode=True)` is your `read_file`, `write_file`, and `desk_path` rule: the agent sees the desk as `/` and cannot reach anything outside it.

In [9]:
show_code(inspect.getsource(build_investigator))

```python
def build_investigator(model, desk_folder: Path):
    """Build the agent. Its file tools can reach the desk folder and nothing else."""
    from deepagents import SubAgent, create_deep_agent  # imported here: a large library
    from deepagents.backends import FilesystemBackend
    from langchain.agents.middleware import TodoListMiddleware

    helper: SubAgent = {
        "name": "investigator",
        "description": "Investigates one finding and writes its report file. Give it the finding id.",
        "system_prompt": HELPER_INSTRUCTIONS,
    }
    return create_deep_agent(
        model=model,
        system_prompt=MAIN_INSTRUCTIONS,
        subagents=[helper],
        middleware=[TodoListMiddleware()],
        backend=FilesystemBackend(root_dir=Path(desk_folder), virtual_mode=True),
    )
```

In [10]:
agent = build_investigator(llm, DESK)

## Step 5: Work the queue

`run_investigator` sends one message, "Please investigate the findings.", and prints every step as it happens.

What to look for in the output:

- `ls` and `read_file` at the start: the agent looking around.
- `write_todos`: the plan, with each item's status.
- `task`: a finding handed to a helper.
- **Indented lines** are a helper's steps: reading its finding, reading the rulebook, writing its report.
- `write_file /handover.md` near the end.

It takes about a minute.

In [11]:
run = run_investigator(agent)

print(f"\nFinished in {run.seconds} seconds: {run.model_calls} model calls, ${run.cost:.4f}")
if run.error:
    print("The run stopped early:", run.error)

ls /
glob **/*
write_todos: Investigate QF-001 (investigator subagen (in_progress), Investigate QF-002 (investigator subagen (in_progress), Investigate QF-003 (investigator subagen (in_progress), Investigate QF-004 (investigator subagen (in_progress), Investigate QF-005 (investigator subagen (in_progress), Investigate QF-006 (investigator subagen (in_progress), Investigate QF-007 (investigator subagen (in_progress), Confirm all 7 .json reports exist in rep (pending), Read reports and rulebook, write handove (pending)
task: QF-001
task: QF-002
task: QF-003
task: QF-004
task: QF-005
task: QF-006
task: QF-007
      glob /findings
      read_file /rulebook.md
      glob /findings
      read_file /rulebook.md
      glob findings/QF-002*
      read_file rulebook.md
      ls /findings
      read_file /rulebook.md
      glob **/*QF-003*
      read_file /rulebook.md
      ls /findings
      read_file /rulebook.md
      glob **/QF-006*
      glob **/rulebook.md
      read_file /findings/QF-001.j

> **If the run stopped early**, run steps 2, 4, and 5 again. The desk is rebuilt fresh each time.

Find the places where it used its to-do list, and where it handed a finding to a helper. Helpers often run several at a time, so their lines can be mixed together.

## Step 6: How did it do?

Each report is a file in `reports/`. Here is one.

In [12]:
print((DESK / "reports" / "QF-003.json").read_text())

{
  "finding_id": "QF-003",
  "likely_cause": "batch_loaded_twice",
  "evidence": "Four order items (OI-01433, OI-01434, OI-01436, OI-01437) are each duplicated across order_items, payment_fees and shipments, which is 12 extra copies. All four orders (ORD-21389, ORD-21390, ORD-21392, ORD-21393) have the same order date, 2026-10-05, so this matches a whole day's file loaded twice.",
  "suggested_action": "Check the 2026-10-05 order load for P002 and remove the second load so the duplicate rows are not kept in the source tables."
}



`read_reports` reads every report on the desk into a table. `score_reports` adds the cause we planted and marks each one.

In [13]:
reports = read_reports(DESK)
scored = score_reports(reports)

pd.set_option("display.max_colwidth", 140)
scored[["finding_id", "likely_cause", "planted_cause", "result"]]

,finding_id,likely_cause,planted_cause,result
0,QF-001,single_record_resent,single_record_resent,right
1,QF-002,single_record_resent,single_record_resent,right
2,QF-003,batch_loaded_twice,batch_loaded_twice,right
3,QF-004,monthly_cost_not_updated,monthly_cost_not_updated,right
4,QF-005,order_recorded_late,order_recorded_late,right
5,QF-006,date_typed_wrong,date_typed_wrong,right
6,QF-007,amount_typed_wrong,amount_typed_wrong,right


In [14]:
harness_right = int((scored["result"] == "right").sum())

print(harness_right, "of", len(scored), "right")

7 of 7 right


Read the evidence it gave. A cause is only useful to a person if they can see why it was chosen.

In [15]:
for row in scored.to_dict("records"):
    print(row["finding_id"], "-", row["evidence"])
    print("   Suggested:", row["suggested_action"])

QF-001 - Only one record is duplicated: refund REF-0014 (copy number 2) for order item OI-00514, which belongs to a single order (ORD-20499) dated 2026-09-13, so no batch of several records from one order date was loaded twice. The rulebook sign for single_record_resent is one duplicated record, which matches.
   Suggested: Check the refund source feed for REF-0014 and stop the system from resending single refund records, keeping the quarantined copy out of the figures.
QF-002 - Three order items (OI-01246, OI-01750, OI-01924) are each duplicated across order_items, payment_fees and shipments, but their orders have different order dates (2026-10-01, 2026-10-13, 2026-10-17), so this is not a single day's file loaded twice. The rulebook's sign for single_record_resent is duplicates tied to orders with different order dates.
   Suggested: Find out why the system re-sent these individual order item, fee and shipment records, fix the resend, and confirm the quarantined copies stay out of th

And the note it left for the data engineer. Look at how it grouped the findings.

In [16]:
display(Markdown((DESK / "handover.md").read_text()))

# Handover: data quality findings QF-001 to QF-007

Kestrel Ridge Outfitters. All seven findings have been investigated and each has a report in `reports/`. This note only records what was found. No data or check outcomes were changed.

## Do first

**QF-004 (Camp stove, P007, 2026-10).** This is the only finding where a figure is withheld. The gross and contribution profit for Camp stove in October are unavailable, and the category total leaves the product out. The cause is that the 2026-10 unit cost was never entered; the 2026-09 cost is 44.0. Entering the October cost and re-running the profit calculation should clear it.

## Findings by cause

**monthly_cost_not_updated**
- QF-004: Camp stove, 2026-10. The October cost was not entered. Action: enter it, then re-run the profit calculation.

**single_record_resent** (system resent single records)
- QF-001: Two-person tent, 2026-09. One refund, REF-0014, was duplicated. Action: check the refund feed and stop single refund records being resent. The extra copy is already quarantined.
- QF-002: Trail water bottle, 2026-10. Nine duplicate rows across order items, fees and shipments for three orders dated 2026-10-01, 10-13 and 10-17. Action: find and fix the resend. Confirm the quarantined copies stay out of the October figures.

**batch_loaded_twice** (a whole day's file loaded twice)
- QF-003: Insulated flask, 2026-10. Twelve duplicate rows for four orders, all dated 2026-10-05. Action: check the 2026-10-05 load for P002 and remove the second load from the source tables. The duplicates are already quarantined.

**order_recorded_late** (order written up a day after it was placed)
- QF-005: Rain jacket, 2026-09. Shipment SHP-00064 is dated 2026-09-01, one day before order ORD-20061 (2026-09-02). Action: confirm the order date with the order system. No change is needed to the shipment date.

**date_typed_wrong** (mistyped date)
- QF-006: Hydration pack, 2026-10. Refund REF-0049 is dated 2026-10-19, 12 days before its order ORD-22432 (2026-10-31). Action: check the refund date against the source documents. The record is still counted until this is fixed.

**amount_typed_wrong** (mistyped amount)
- QF-007: Sun hat, 2026-09. Refund REF-0002 is 60.00 against a line worth 25.00 (one line, no discount, on order ORD-20077). Action: check the refund amount against the source documents. Net sales may be understated while the record is counted.

## Notes
- QF-001, QF-002 and QF-003 share a check (duplicate records) and a loading problem. They are quarantined, so the figures are not affected. The root cause still needs fixing.
- QF-005, QF-006 and QF-007 are records that are still counted. Their figures may be wrong until someone checks the source documents.
- Each finding has a suggested action in its report. Those actions have not been carried out.


## Step 7: Read the prompt before you blame the model

When this lab was built, the first version of the main agent's instructions did not say what a report should look like. Only the helper's instructions did.

The result on the first model tried: **0 of 7**. A second model produced readable reports for only three findings. The main agent, left to guess, invented its own report format (a long `.md` file) and told each helper to use it. The helpers did as they were told. The reasoning inside those files was good. Code could not read any of it.

One sentence fixed it:

> *Tell it only the finding id. It already knows how to investigate a finding, and it writes the report to reports/<finding id>.json.*

After that, three different models each scored 7 of 7. When more than one model fails in the same way, suspect your instructions first.

## Step 8: A report that cannot be used

Models do not always do what they are told. What happens when a report is missing or broken? Let's damage three on purpose.

In [17]:
(DESK / "reports" / "QF-001.json").unlink()

(DESK / "reports" / "QF-002.json").write_text("I think the cause is probably a retry.")

report = json.loads((DESK / "reports" / "QF-003.json").read_text())
report["likely_cause"] = "cosmic_rays"
(DESK / "reports" / "QF-003.json").write_text(json.dumps(report))

print("Damaged three reports.")

Damaged three reports.


One report is gone, one is not JSON, and one names a cause that is not in the rulebook. Read and score them again.

In [18]:
damaged = score_reports(read_reports(DESK))

damaged[["finding_id", "status", "problem", "result"]]

,finding_id,status,problem,result
0,QF-001,left for a person,No report was written.,left for a person
1,QF-002,left for a person,The report is not valid JSON.,left for a person
2,QF-003,left for a person,'cosmic_rays' is not a cause in the rulebook.,left for a person
3,QF-004,report read,,right
4,QF-005,report read,,right
5,QF-006,report read,,right
6,QF-007,report read,,right


Each one is marked **left for a person**, with the reason. The other four are unaffected. Code never guesses what a broken report meant, and one bad report never stops the rest.

## Step 9: What the agent could not do

The agent wrote eight files. Did any of that change the data? Look at the findings table the pipeline wrote.

In [19]:
findings, finding_records = read_findings()

findings[["finding_id", "check_name", "outcome", "severity"]]

,finding_id,check_name,outcome,severity
0,QF-001,duplicate_records,quarantine,low
1,QF-002,duplicate_records,quarantine,low
2,QF-003,duplicate_records,quarantine,low
3,QF-004,missing_product_cost,needs_review,high
4,QF-005,event_out_of_order,needs_review,low
5,QF-006,event_out_of_order,needs_review,low
6,QF-007,refund_exceeds_line,needs_review,medium


Unchanged, and it has to be:

- The agent's file tools reach only the `desk` folder, which holds **copies**.
- No code reads an outcome or a figure from a report. The pipeline, the tools, and the analyst never open them.

That is the rule from milestone 1 again: the model explains, and code decides.

## Step 10: Three ways, side by side

In [20]:
earlier = load_result("by_hand", required_from="notebook 4")

pd.DataFrame([
    {"approach": "One model call", "tools we wrote": 0, "findings": earlier["one_call"]["findings"], "right": earlier["one_call"]["right"], "model calls": earlier["one_call"]["model_calls"]},
    {"approach": "Agent built by hand", "tools we wrote": 5, "findings": earlier["by_hand"]["findings"], "right": earlier["by_hand"]["right"], "model calls": earlier["by_hand"]["model_calls"]},
    {"approach": "Harness (Deep Agents)", "tools we wrote": 0, "findings": len(scored), "right": harness_right, "model calls": run.model_calls},
])

,approach,tools we wrote,findings,right,model calls
0,One model call,0,7,7,1
1,Agent built by hand,5,3,3,19
2,Harness (Deep Agents),0,7,7,38


On seven findings, all three usually get the causes right. The differences are elsewhere.

| | One model call | Agent built by hand | Harness |
|---|---|---|---|
| Model calls | 1 | Several per finding | Several per finding |
| Works when the queue is long | No: the prompt grows | Yes | Yes |
| One failure loses everything | Yes | No | No |
| You can see each step | No | Yes | Yes |
| Code you maintain | A prompt | The tools | Two sets of instructions |

The harness costs more calls than one prompt. That is the price of working one finding at a time. For seven findings, one call is the sensible choice. For seven hundred, it is not an option.

**Build or buy?** The harness gave you a capable agent for no tool code. In return you depend on a library, and you see less of what it does. You built the same thing by hand first so that you know what is inside.

> **Your assignment.** Try another model and compare the right answers, the cost, and the time. Change the first cell to, for example, `MODEL = "google/gemma-4-31b-it"` or `MODEL = "deepseek/deepseek-v4.1-flash"` after the `connect()` line, then run steps 2 to 6 again.

## What we now know

1. A harness gives an agent files, a to-do list, and helpers ready-made. You write instructions, not tools.
2. An agent's work must be scored. We could, because we planted the causes.
3. A report that is missing or broken is left for a person. Code never guesses.
4. The agent cannot change the data or an outcome. Its reports are notes.

**Next:** notebook 6. The leader asks the same question, and this time the warnings are real.